# 面试问题：SimPO 为什么无需 reference model，长度归一化隐式 reward、target margin 与稳定 loss 如何从零实现？

**一句话回答。** SimPO 直接把当前策略对回答的“平均 token 对数概率”乘以缩放系数 $\beta$ 当作隐式 reward：

$$r_{\text{SimPO}}(x,y)=\frac{\beta}{|y|}\log\pi_\theta(y\mid x).$$

对偏好对 $(y_w,y_l)$，再优化 $-\log\sigma(r_w-r_l-\gamma)$。这里没有 DPO 中的策略/参考策略对数概率比，因此训练时不必常驻一份冻结 reference model；$1/|y|$ 让 reward 与逐 token 平均生成概率对齐；$\gamma$ 要求 chosen 不只是略胜，而是至少形成目标间隔。

本 Notebook 用基础 PyTorch 张量和 `nn.Module` 手写稳定 log-softmax、响应 mask、SimPO reward 与 loss，并用标量梯度和一个微型因果模型验证优化方向。它是公式级教学实验，不等价于论文规模复现或线上对齐效果证明。

**主要资料。** [SimPO: Simple Preference Optimization with a Reference-Free Reward](https://arxiv.org/abs/2405.14734)（NeurIPS 2024）。

In [ ]:
import torch  # 导入 PyTorch 张量、自动微分与数值计算能力。
from torch import nn  # 导入神经网络基础模块以手写微型因果语言模型。
torch.manual_seed(237)  # 固定随机种子，让初始化和训练结果可以重复检查。
beta = 2.0  # 设置隐式 reward 的缩放系数，控制偏好梯度强度。
target_margin = 0.5  # 设置 chosen reward 应领先 rejected reward 的目标间隔。
assert torch.__version__  # 确认当前环境已经成功载入 PyTorch。
assert beta > 0.0  # 验证 reward 缩放系数必须为正数。
assert target_margin >= 0.0  # 验证目标间隔不取负值。
assert torch.initial_seed() == 237  # 验证实验确实使用了约定的随机种子。

## 1. 从 DPO 的比值到 reference-free reward

DPO 的判别量包含 $\log\pi_\theta(y\mid x)-\log\pi_{\text{ref}}(y\mid x)$，reference model 提供相对基准，也意味着训练时要计算或缓存参考策略的序列分数。SimPO 改用当前策略本身的平均 log probability 作为 reward，因此偏好对的 logit 只依赖一次可训练策略的 chosen/rejected 前向。

“无需 reference model”不等于“没有正则化，也不可能漂移”。它只描述目标函数不显式依赖冻结参考策略。生产训练仍要通过学习率、数据质量、SFT 混合、KL 监控、早停与安全评测约束分布漂移。面试时应把“省去参考模型的算力/显存”与“完全不需要行为约束”清楚地区分。

In [ ]:
def stable_softplus(value):  # 定义数值稳定的 softplus，避免直接计算大指数造成溢出。
    return torch.clamp(value, min=0.0) + torch.log1p(torch.exp(-torch.abs(value)))  # 使用最大值分解手写 log(1+exp(x)) 的稳定形式。
def stable_negative_log_sigmoid(logit):  # 定义 SimPO 单样本的负对数 sigmoid 损失。
    return stable_softplus(-logit)  # 利用 -log(sigmoid(z)) 等价于 softplus(-z)。
extreme_logits = torch.tensor([-1000.0, 0.0, 1000.0])  # 构造极负、零和极正的偏好 logit 检查稳定性。
extreme_losses = stable_negative_log_sigmoid(extreme_logits)  # 计算三个极端输入对应的稳定损失。
assert torch.isfinite(extreme_losses).all()  # 验证极端 logit 下没有出现无穷大或非数值。
assert torch.allclose(extreme_losses[1], torch.log(torch.tensor(2.0)))  # 验证零 logit 的损失等于 log(2)。
assert extreme_losses[0] > extreme_losses[1] > extreme_losses[2]  # 验证 chosen 越占优时损失越小。
assert extreme_losses[2] < 1e-6  # 验证极大正间隔已经对应近似为零的惩罚。

## 2. 长度归一化到底解决什么

自回归序列的联合 log probability 是各响应 token 条件 log probability 的和。因为每项通常为负数，回答越长，求和往往越负；若直接把总和当 reward，长度就会成为强混杂因素。SimPO 除以响应 token 数，只比较每个有效响应 token 的平均 log probability，使目标和推理解码常用的长度归一化分数更一致。

注意分母只能统计回答区域：prompt、padding、被忽略标签都不能进入分子或分母。长度归一化也不是“绝不偏好长答案”的数学保证，因为内容、EOS 概率、数据分布和解码器仍会影响长度；所以训练后仍要分桶报告长度、胜率和 reward gap。

In [ ]:
def manual_log_softmax(logits):  # 手写稳定的 log-softmax，展示 token 对数概率如何得到。
    row_max = logits.max(dim=-1, keepdim=True).values  # 取每个词表行最大值以消除指数溢出风险。
    shifted = logits - row_max  # 平移 logits 而不改变 softmax 概率。
    return shifted - torch.log(torch.exp(shifted).sum(dim=-1, keepdim=True))  # 用移位后的 log-sum-exp 得到归一化对数概率。
def masked_average_log_probability(logits, targets, response_mask):  # 仅在回答 token 上计算每条序列的平均 log probability。
    token_log_probs = manual_log_softmax(logits).gather(-1, targets.unsqueeze(-1)).squeeze(-1)  # 取出真实目标 token 的条件对数概率。
    token_counts = response_mask.sum(dim=-1).clamp_min(1)  # 统计每条回答有效 token 数并防止零除。
    averages = (token_log_probs * response_mask).sum(dim=-1) / token_counts  # 对响应区域求和后按有效长度归一化。
    return averages, token_log_probs  # 同时返回序列均值和逐 token 分数以便诊断。
toy_logits = torch.tensor([[[2.0, 0.0, -1.0], [0.0, 2.0, -1.0], [-1.0, 0.0, 2.0]]])  # 构造一条三步词表分布作为可核验样例。
toy_targets = torch.tensor([[0, 1, 2]])  # 指定每一步正确 token 都是该行最高分项。
toy_mask = torch.tensor([[0.0, 1.0, 1.0]])  # 屏蔽第一步并只把后两步视为回答区域。
toy_average, toy_token_log_probs = masked_average_log_probability(toy_logits, toy_targets, toy_mask)  # 计算响应区域的长度归一化序列分数。
assert toy_token_log_probs.shape == toy_targets.shape  # 验证逐 token 对数概率与标签形状一致。
assert toy_average.shape == torch.Size([1])  # 验证每条序列只得到一个平均分数。
assert torch.allclose(toy_average, toy_token_log_probs[:, 1:].mean(dim=-1))  # 验证 prompt 位置确实没有进入平均值。
assert torch.allclose(manual_log_softmax(toy_logits).exp().sum(dim=-1), torch.ones(1, 3))  # 验证手写归一化后的概率逐行和为一。
assert toy_average.item() < 0.0  # 验证合法概率的自然对数均值不大于零。

## 3. 不导入 Transformer：手写一个可训练的微型因果模型

为了把重点放在目标函数，本例用 `Embedding → 前缀累积均值 → Linear` 构造微型因果语言模型。位置 $t$ 的表示只汇总不晚于 $t$ 的 token，因此它可以预测下一个 token，未来 token 不会泄露。这个结构不具备注意力、位置编码和真实大模型容量，但保留了“每个位置输出下一 token 词表 logits、参数通过序列 loss 更新”的必要接口。

面试实现的关键不是把 `AutoModelForCausalLM` 换个名字，而是能够说明 logits 与 labels 为什么错开一位、回答 mask 为什么从 prompt 末尾开始，以及参数梯度怎样从 preference loss 反传到 token 分布。

In [ ]:
class TinyCausalLM(nn.Module):  # 定义一个不依赖高层语言模型库的微型因果模型。
    def __init__(self, vocabulary_size, hidden_size):  # 接收词表规模和隐藏维度完成参数初始化。
        super().__init__()  # 初始化 PyTorch 模块基类以正确登记参数。
        self.embedding = nn.Embedding(vocabulary_size, hidden_size)  # 为每个离散 token 学习一个隐藏向量。
        self.output_projection = nn.Linear(hidden_size, vocabulary_size, bias=False)  # 将因果前缀表示投影为下一 token logits。
    def forward(self, input_ids):  # 根据输入 token 序列计算每个位置的下一 token 分布。
        embeddings = self.embedding(input_ids)  # 把 token 编号映射成连续隐藏向量。
        prefix_sums = torch.cumsum(embeddings, dim=1)  # 累积当前位置及其之前的向量以保持因果性。
        prefix_lengths = torch.arange(1, input_ids.shape[1] + 1, device=input_ids.device, dtype=embeddings.dtype).view(1, -1, 1)  # 构造每个位置实际聚合的前缀长度。
        causal_hidden = prefix_sums / prefix_lengths  # 用前缀均值得到尺度较稳定的因果隐藏状态。
        return self.output_projection(causal_hidden)  # 输出批次、时间步和词表三维 logits。
probe_model = TinyCausalLM(vocabulary_size=10, hidden_size=8)  # 创建一个小模型用于检查接口和因果性。
probe_inputs = torch.tensor([[1, 2, 3, 4], [1, 2, 7, 8]])  # 构造前两个 token 相同而后缀不同的两条序列。
probe_logits = probe_model(probe_inputs)  # 执行一次前向传播得到每个时间步的词表 logits。
assert probe_logits.shape == torch.Size([2, 4, 10])  # 验证输出形状符合自回归语言模型约定。
assert torch.isfinite(probe_logits).all()  # 验证随机初始化前向没有非数值。
assert torch.allclose(probe_logits[0, :2], probe_logits[1, :2])  # 验证未来不同后缀不会改变共同前缀位置的输出。
assert sum(parameter.numel() for parameter in probe_model.parameters()) == 160  # 验证参数只来自嵌入矩阵和输出投影矩阵。

## 4. 偏好批次：prompt 相同，回答与 mask 分开

每条数据包含同一 prompt 下的 chosen 与 rejected。模型输入是完整序列去掉最后一个 token，label 是完整序列去掉第一个 token；因此第 $t$ 个输入位置预测第 $t+1$ 个 token。若 prompt 长度为 $P$，响应 mask 应从 label 下标 $P-1$ 开始，因为那个位置预测回答的第一个 token。

chosen 与 rejected 长度可以不同，所以两支分别 padding，并分别计算有效 token 数。padding 只服务于批处理，既不能计入 reward，也不能通过错误 label 影响损失。真实工程还需处理截断：至少保留 EOS，并记录被截断样本，否则模型可能学到不完整回答的伪偏好。

In [ ]:
vocabulary = {"<pad>": 0, "<bos>": 1, "问题": 2, "清楚": 3, "准确": 4, "步骤": 5, "含糊": 6, "错误": 7, "冗余": 8, "结论": 9}  # 定义微型实验所需的离散词表。
prompt_tokens = [vocabulary["<bos>"], vocabulary["问题"]]  # 构造两枚 token 的共同 prompt。
chosen_responses = [[vocabulary["清楚"], vocabulary["准确"]], [vocabulary["步骤"], vocabulary["清楚"], vocabulary["结论"]]]  # 构造两条长度不同的优选回答。
rejected_responses = [[vocabulary["含糊"]], [vocabulary["冗余"], vocabulary["错误"]]]  # 构造对应的两条劣选回答。
def pack_response_batch(prompt, responses, pad_id):  # 把可变长回答打包成下一 token 预测批次和响应 mask。
    maximum_steps = max(len(prompt) + len(response) - 1 for response in responses)  # 计算移位后需要容纳的最大时间步数。
    inputs = torch.full((len(responses), maximum_steps), pad_id, dtype=torch.long)  # 初始化 padding 后的模型输入张量。
    targets = torch.full((len(responses), maximum_steps), pad_id, dtype=torch.long)  # 初始化与输入错开一位的目标张量。
    response_mask = torch.zeros((len(responses), maximum_steps), dtype=torch.float32)  # 初始化只标记回答 token 的浮点 mask。
    for row, response in enumerate(responses):  # 逐条处理不同长度的候选回答。
        complete_sequence = prompt + response  # 拼接共同 prompt 与当前回答得到完整序列。
        prediction_steps = len(complete_sequence) - 1  # 计算移位后真实存在的下一 token 预测步数。
        inputs[row, :prediction_steps] = torch.tensor(complete_sequence[:-1])  # 写入去掉末 token 的模型输入。
        targets[row, :prediction_steps] = torch.tensor(complete_sequence[1:])  # 写入去掉首 token 的下一 token 标签。
        response_mask[row, len(prompt) - 1:prediction_steps] = 1.0  # 从预测首个回答 token 的位置开始打开 mask。
    return inputs, targets, response_mask  # 返回模型输入、目标标签与响应区域 mask。
chosen_inputs, chosen_targets, chosen_mask = pack_response_batch(prompt_tokens, chosen_responses, vocabulary["<pad>"])  # 打包优选回答分支。
rejected_inputs, rejected_targets, rejected_mask = pack_response_batch(prompt_tokens, rejected_responses, vocabulary["<pad>"])  # 打包劣选回答分支。
assert chosen_inputs.shape == chosen_targets.shape == chosen_mask.shape  # 验证优选分支的输入、标签和 mask 完全对齐。
assert rejected_inputs.shape == rejected_targets.shape == rejected_mask.shape  # 验证劣选分支的三个张量完全对齐。
assert chosen_mask.sum(dim=-1).tolist() == [2.0, 3.0]  # 验证优选分母等于各自回答 token 数。
assert rejected_mask.sum(dim=-1).tolist() == [1.0, 2.0]  # 验证劣选分母等于各自回答 token 数。
assert not chosen_mask[:, :len(prompt_tokens) - 1].bool().any()  # 验证共同 prompt 的预测位置没有混入隐式 reward。

## 5. 从公式直接实现 SimPO objective

先分别得到 chosen 与 rejected 的响应平均 log probability，乘 $\beta$ 后形成 $r_w,r_l$。偏好 logit 为 $z=r_w-r_l-\gamma$，批次损失为 $\operatorname{mean}(\operatorname{softplus}(-z))$。减去 $\gamma$ 意味着 reward 差尚未达到目标间隔时，样本仍会受到明显惩罚。

$\beta$ 与 $\gamma$ 作用不同：$\beta$ 同时缩放两个序列分数及其梯度，$\gamma$ 平移判别边界。实践中不能只看 loss，要同时记录 chosen/rejected reward、原始 gap、满足 margin 的比例、回答长度与梯度范数；否则同一个平均 loss 可能掩盖少量困难或标注冲突样本。

In [ ]:
def average_response_log_probability(model, inputs, targets, response_mask):  # 计算当前策略对一批回答的平均 token 对数概率。
    logits = model(inputs)  # 让同一个可训练策略对输入序列执行前向传播。
    averages, token_log_probs = masked_average_log_probability(logits, targets, response_mask)  # 使用响应 mask 聚合有效 token 分数。
    return averages, token_log_probs  # 返回序列平均值和逐 token 诊断值。
def simpo_loss(chosen_average, rejected_average, reward_scale, margin):  # 按论文公式从两个平均序列分数构造 SimPO 损失。
    chosen_reward = reward_scale * chosen_average  # 把优选回答平均 log probability 缩放成隐式 reward。
    rejected_reward = reward_scale * rejected_average  # 把劣选回答平均 log probability 缩放成隐式 reward。
    raw_reward_gap = chosen_reward - rejected_reward  # 计算尚未扣除目标间隔的 reward 差。
    preference_logit = raw_reward_gap - margin  # 扣除目标 margin 后得到 Bradley-Terry 判别 logit。
    per_pair_loss = stable_negative_log_sigmoid(preference_logit)  # 用稳定 softplus 形式计算每个偏好对的损失。
    return per_pair_loss.mean(), {"chosen_reward": chosen_reward, "rejected_reward": rejected_reward, "raw_gap": raw_reward_gap, "logit": preference_logit}  # 返回批次均值和可审计指标。
initial_model = TinyCausalLM(vocabulary_size=len(vocabulary), hidden_size=12)  # 创建待验证的当前策略且不创建任何参考模型。
initial_chosen_average, _ = average_response_log_probability(initial_model, chosen_inputs, chosen_targets, chosen_mask)  # 计算初始化策略的优选回答平均分。
initial_rejected_average, _ = average_response_log_probability(initial_model, rejected_inputs, rejected_targets, rejected_mask)  # 计算初始化策略的劣选回答平均分。
initial_loss, initial_metrics = simpo_loss(initial_chosen_average, initial_rejected_average, beta, target_margin)  # 计算初始化策略的 SimPO loss 与指标。
assert initial_loss.ndim == 0  # 验证批次损失已经归约成标量。
assert torch.isfinite(initial_loss)  # 验证初始损失为有限数值。
assert initial_metrics["chosen_reward"].shape == torch.Size([2])  # 验证每个偏好对都有独立 chosen reward。
assert torch.allclose(initial_metrics["logit"], initial_metrics["raw_gap"] - target_margin)  # 验证 margin 正确从原始 reward gap 中扣除。
assert len(list(initial_model.parameters())) == 2  # 验证目标只依赖当前模型的嵌入与投影参数而没有冻结参考模型。

## 6. 梯度方向：为什么 chosen 上升、rejected 下降

把两个序列平均分先视为独立标量。对于 $L=\operatorname{softplus}(-(\beta s_w-\beta s_l-\gamma))$，当样本尚未满足间隔时，$\partial L/\partial s_w<0$、$\partial L/\partial s_l>0$。梯度下降因此提高 chosen 的平均 log probability、降低 rejected 的平均 log probability。

两者梯度绝对值对称是标量公式的性质，但共享语言模型参数后，两个回答可能复用 prompt、token 与隐藏状态，最终参数更新不会简单分解成两个独立旋钮。用标量实验确认符号，再用真实共享模型确认总体 reward gap，能快速定位 mask 反了、chosen/rejected 对调或 margin 符号写错等常见 bug。

In [ ]:
chosen_score = torch.tensor(-2.0, requires_grad=True)  # 把优选回答平均 log probability 建成可求导标量。
rejected_score = torch.tensor(-2.0, requires_grad=True)  # 把劣选回答平均 log probability 建成可求导标量。
scalar_loss, scalar_metrics = simpo_loss(chosen_score.view(1), rejected_score.view(1), beta, target_margin)  # 在两者初始相等时计算带 margin 的损失。
scalar_loss.backward()  # 通过手写目标函数反向传播得到两个分数的梯度。
zero_margin_loss, _ = simpo_loss(chosen_score.detach().view(1), rejected_score.detach().view(1), beta, 0.0)  # 计算同一分数下没有目标间隔的对照损失。
assert chosen_score.grad.item() < 0.0  # 验证梯度下降会提高 chosen 平均 log probability。
assert rejected_score.grad.item() > 0.0  # 验证梯度下降会降低 rejected 平均 log probability。
assert torch.allclose(chosen_score.grad, -rejected_score.grad)  # 验证独立标量情形下两支梯度大小相同而方向相反。
assert scalar_loss > zero_margin_loss  # 验证正 margin 会提高尚未拉开差距样本的训练压力。
assert scalar_metrics["logit"].item() == -target_margin  # 验证两分数相等时判别 logit 恰好是负目标间隔。

## 7. 用同一个策略完成小训练闭环

下面重新初始化一个策略，对两组 preference pair 做少量 Adam 更新。每一步只运行当前模型的 chosen/rejected 分支，二者共享全部参数；没有 reference logits、reference checkpoint 或参考模型前向。实验验收标准不是生成自然语言，而是 loss 下降、平均 reward gap 上升、参数确实变化，并且最终每个 pair 都把 chosen 排在 rejected 之前。

小数据很容易被记忆，因此这里的成功只证明代码和梯度方向自洽。真实训练要划分留出集，监控过拟合与长度分布，并与 SFT、DPO 等基线在相同数据、解码和评测协议下比较。

In [ ]:
torch.manual_seed(237)  # 再次固定种子以独立复现实训模型初始化。
trained_model = TinyCausalLM(vocabulary_size=len(vocabulary), hidden_size=12)  # 创建唯一参与优化的当前策略模型。
optimizer = torch.optim.Adam(trained_model.parameters(), lr=0.05)  # 使用基础 Adam 优化器而不依赖任何 Trainer 封装。
starting_projection = trained_model.output_projection.weight.detach().clone()  # 保存训练前权重用于验证参数发生更新。
with torch.no_grad():  # 在不构建梯度图的条件下记录训练前基线指标。
    before_chosen, _ = average_response_log_probability(trained_model, chosen_inputs, chosen_targets, chosen_mask)  # 记录训练前 chosen 平均分。
    before_rejected, _ = average_response_log_probability(trained_model, rejected_inputs, rejected_targets, rejected_mask)  # 记录训练前 rejected 平均分。
    before_loss, before_metrics = simpo_loss(before_chosen, before_rejected, beta, target_margin)  # 记录训练前损失与 reward gap。
loss_history = []  # 创建列表保存每一步损失以检查优化趋势。
for training_step in range(80):  # 在微型偏好数据上执行有限步数的确定性训练。
    optimizer.zero_grad()  # 清除上一步累积在模型参数上的梯度。
    chosen_average, _ = average_response_log_probability(trained_model, chosen_inputs, chosen_targets, chosen_mask)  # 用当前策略计算 chosen 平均 token 对数概率。
    rejected_average, _ = average_response_log_probability(trained_model, rejected_inputs, rejected_targets, rejected_mask)  # 用同一策略计算 rejected 平均 token 对数概率。
    training_loss, training_metrics = simpo_loss(chosen_average, rejected_average, beta, target_margin)  # 根据 reference-free 公式计算本步损失。
    training_loss.backward()  # 将偏好损失梯度反传到嵌入和输出投影参数。
    optimizer.step()  # 使用 Adam 根据当前梯度更新唯一策略模型。
    loss_history.append(training_loss.detach().item())  # 保存脱离计算图的标量损失用于诊断。
with torch.no_grad():  # 在训练结束后关闭自动微分并计算最终指标。
    after_chosen, _ = average_response_log_probability(trained_model, chosen_inputs, chosen_targets, chosen_mask)  # 计算训练后 chosen 平均分。
    after_rejected, _ = average_response_log_probability(trained_model, rejected_inputs, rejected_targets, rejected_mask)  # 计算训练后 rejected 平均分。
    after_loss, after_metrics = simpo_loss(after_chosen, after_rejected, beta, target_margin)  # 计算训练后损失和 reward gap。
assert after_loss.item() < before_loss.item()  # 验证小训练闭环确实降低 SimPO 目标。
assert after_metrics["raw_gap"].mean() > before_metrics["raw_gap"].mean()  # 验证平均隐式 reward 差朝正确方向扩大。
assert (after_chosen > after_rejected).all()  # 验证最终每个偏好对都把 chosen 平均分排在 rejected 之前。
assert not torch.allclose(starting_projection, trained_model.output_projection.weight.detach())  # 验证输出投影参数确实被梯度更新。

## 8. 稳定性、监控与工程边界

稳定实现至少包含四层检查：数值层确认极端 logit 的 loss/gradient 有限；数据层确认 prompt、padding、截断与 EOS mask；优化层监控两支 reward、gap、margin 命中率和梯度；评测层按长度分桶报告留出偏好准确率与生成质量。若只报告训练 loss，可能把标签噪声、长度漂移或通用能力退化隐藏起来。

reference-free 节省的是显式参考模型的前向与驻留成本，但 chosen/rejected 仍需要两支当前策略计算。工程上可把两支样本拼成一个 batch 提高吞吐，还要考虑混合精度、梯度累积、分布式聚合和超长序列。这里使用 float32 小张量，不能据此声称 bf16、FlashAttention 或多机实现稳定。

In [ ]:
with torch.no_grad():  # 关闭梯度并汇总可用于训练监控的最终指标。
    final_raw_gaps = after_metrics["raw_gap"]  # 读取每个偏好对扣除 margin 前的 reward 差。
    margin_hit_rate = (final_raw_gaps > target_margin).float().mean()  # 统计达到目标 reward 间隔的样本比例。
    preference_accuracy = (after_chosen > after_rejected).float().mean()  # 统计 chosen 平均分高于 rejected 的比例。
    monitoring_report = {"final_loss": after_loss.item(), "mean_reward_gap": final_raw_gaps.mean().item(), "margin_hit_rate": margin_hit_rate.item(), "preference_accuracy": preference_accuracy.item(), "reference_model_count": 0}  # 汇总面试中应解释的核心诊断字段。
stress_gap = torch.tensor([-10000.0, 10000.0], requires_grad=True)  # 构造极端正负 reward gap 检查稳定梯度。
stress_loss = stable_negative_log_sigmoid(stress_gap - target_margin).mean()  # 使用同一稳定公式计算压力测试损失。
stress_loss.backward()  # 对极端输入执行反向传播验证梯度不会成为非数值。
assert torch.isfinite(stress_loss)  # 验证极端 reward gap 下总损失仍为有限值。
assert torch.isfinite(stress_gap.grad).all()  # 验证极端 reward gap 下梯度仍为有限值。
assert monitoring_report["preference_accuracy"] == 1.0  # 验证微型训练集上的偏好排序全部正确。
assert monitoring_report["margin_hit_rate"] == 1.0  # 验证微型训练集最终全部超过指定目标间隔。
assert monitoring_report["reference_model_count"] == 0  # 明确验证整个目标与训练闭环没有构造参考模型。

## 面试总结

回答这道题可以按五步展开：第一，写出 $r=\beta\log\pi(y|x)/|y|$，说明 SimPO 的 reward 只来自当前策略，所以目标函数不需要显式 reference model；第二，解释响应 mask 与长度归一化消除总 log probability 的直接长度尺度；第三，写出 $z=r_w-r_l-\gamma$ 和稳定的 `softplus(-z)`；第四，用梯度符号说明 chosen 上升、rejected 下降，margin 让“略胜”仍受惩罚；第五，补充 reference-free 不等于无漂移约束，并给出长度分桶、margin 命中率、reward gap、留出集胜率和安全评测。

常见失分点包括：把 prompt token 计入分母、对 padding 求均值、忘记 logits/labels 移位、把 $+\gamma$ 写成 $-\gamma$ 的反方向、直接计算 `log(sigmoid(z))` 导致极端值溢出，以及因为省掉 reference model 就宣称训练成本减半或稳定性自动得到保证。上面的断言分别覆盖了这些公式与实现边界，但真正的模型结论必须由论文规模数据和严格基线实验支持。